In [7]:
## Script to extract the V1 10x data from the merged file using the published_2022 column for generating a TSP V1 matrix

import os
import glob
import scanpy as sc

# Define input and output directories
input_dir ='01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/'
output_dir ='COO-Matrix/TSP-V1'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Find all H5 files in the specified input directory
h5ad_files = glob.glob(os.path.join(input_dir, "*_modified.h5ad"))

for h5ad_file in h5ad_files:
    print(f"Processing {h5ad_file}...")

    # Load the AnnData object
    adata = sc.read_h5ad(h5ad_file)

    # Filter based on method and whether it was previously published
    version_filtered_data = adata[(adata.obs['published_2022'] == 'True') & (adata.obs['method'] == '10X')].copy()

    if version_filtered_data.n_obs > 0:
        # Extract the tissue name before "_TSP"
        base_filename = os.path.splitext(os.path.basename(h5ad_file))[0]
        tissue = base_filename.split('_TSP', 1)[0]
        
        # Construct output filename
        output_filename = f"{tissue}_V1_10X.h5ad"
        output_filename = os.path.join(output_dir, output_filename)
        # Save modified data for each donor
        version_filtered_data.write(output_filename)
        print(f"Saved modified data for {h5ad_file} to: {output_filename}")

Processing 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Bladder_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad...
Saved modified data for 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Bladder_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad to: COO-Matrix/TSP-V1/Bladder_V1_10X.h5ad
Processing 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Blood_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad...
Saved modified data for 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Blood_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad to: COO-Matrix/TSP-V1/Blood_V1_10X.h5ad
Processing 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Bone_Marrow_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad...
Saved modified data for 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Bone_Marrow_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad to: COO-Matrix/TSP-V1/Bone_Marrow_V1_10X.h5ad
Processing 01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/Ear_TSP1_30_version2d_10X_smartseq_scvi_N

## This needs to be improved as donors from the TS1 are maintained in the TS2

import os
import glob
import scanpy as sc

# Define input and output directories
input_dir ='01_TSP_2.0_Figshare-h5ad/01_Modified-h5ad/'
output_dir ='COO-Matrix/TSP-V2'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Find all H5 files in the specified input directory
h5ad_files = glob.glob(os.path.join(input_dir, "*_modified.h5ad"))

for h5ad_file in h5ad_files:
    print(f"Processing {h5ad_file}...")

    # Load the AnnData object
    adata = sc.read_h5ad(h5ad_file)

    # Filter based on method and whether it was previously published
    version_filtered_data = adata[(adata.obs['published_2022'] == 'False') & (adata.obs['method'] == '10X')].copy()

    if version_filtered_data.n_obs > 0:
        # Extract the tissue name before "_TSP"
        base_filename = os.path.splitext(os.path.basename(h5ad_file))[0]
        tissue = base_filename.split('_TSP', 1)[0]
        
        # Construct output filename
        output_filename = f"{tissue}_V2_10X.h5ad"
        output_filename = os.path.join(output_dir, output_filename)
        # Save modified data for each donor
        version_filtered_data.write(output_filename)
        print(f"Saved modified dat   a for {h5ad_file} to: {output_filename}")

In [101]:
## Script to filter and QC the TSP V1 matrix and merge them all in one
import scanpy as sc
import numpy as np

input_dir = 'COO-Matrix/TSP-V1/'

# Find all H5 files in the specified input directory
h5ad_files = glob.glob(os.path.join(input_dir, "*10X.h5ad"))

for h5ad_file in h5ad_files:
    print(f"Processing {h5ad_file}...")

    # Load the data using Scanpy's function for 10X Genomics h5 files
    adata = sc.read_h5ad(h5ad_file)
    adata.X = adata.layers["decontXcounts"].copy()

    # Compute quality control metrics and log-transform counts
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, log1p=False)

    # Identify mitochondrial genes
    mito_genes = adata.var_names.str.startswith("MT-")  # Adjust based on annotation
    non_mito_genes = (adata.X[:, ~mito_genes] > 0).sum(axis=1).A1 if isinstance(adata.X, np.matrix) else (adata.X[:, ~mito_genes] > 0).sum(axis=1)
    adata.obs["non_mito_genes"] = non_mito_genes
    
    # Compute non-mitochondrial counts
    adata.obs["non_mito_counts"] = adata.obs["total_counts"] - adata.obs["total_counts_mt"]

    # Apply Scrublet to detect and annotate doublets in the dataset
    sc.pp.scrublet(adata)

    # Filter cells that are not predicted doublets and have at least 2500 non-mitochondrial counts and 200 detected non-mitochondrial genes
    adata = adata[(adata.obs["non_mito_counts"] >= 2500) & (adata.obs["non_mito_genes"] >= 200) & (adata.obs["predicted_doublet"] == False)].copy()
        
    # Construct the output path with the specified output directory and suffix "_processed.h5ad"
    output_path = os.path.join(input_dir, os.path.basename(h5ad_file).replace(".h5ad", "_processed.h5ad"))

    # Save the processed AnnData object for future analysis
    adata.write(output_path)

Processing COO-Matrix/TSP-V1/Bladder_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Blood_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Bone_Marrow_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Eye_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Fat_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Heart_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Large_Intestine_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Liver_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Lung_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Lymph_Node_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Mammary_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Muscle_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Pancreas_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Prostate_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Salivary_Gland_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Skin_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Small_Intestine_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Spleen_V1_10X.h5ad...
Processing COO-Matrix/TSP-V1/Thymus_V1_10X.h5ad...
Pro

In [102]:
import scanpy as sc
import os
import glob

input_dir = "COO-Matrix/TSP-V1/"

# Find all .h5ad files in the input directory
h5ad_files = glob.glob(os.path.join(input_dir, "*_processed.h5ad"))

# Merge directly without storing all in memory
adata_merged = sc.concat(
    (sc.read_h5ad(f) for f in h5ad_files),  # Generator to avoid full list storage
    join="outer",
    merge="same",
    label="batch",
    keys=[os.path.basename(f) for f in h5ad_files]  # Store batch info from filenames
)

# Save the merged object
output_file = os.path.join(input_dir, "merged_TSP_V1.h5ad")
adata_merged.write_h5ad(output_file)

print(f"Merged dataset saved to: {output_file}")


Merged dataset saved to: COO-Matrix/TSP-V1/merged_TSP_V1.h5ad
